# Uni-MuMER - Kaggle 2xT4 + DagsHub MLflow

Train Uni-MuMER trên Kaggle với:
- Miniconda → Python 3.10 → Train QLoRA 4-bit
- **DagsHub MLflow** tự động đồng bộ logs + checkpoint (~643MB) lên cloud
- Mặc định 2 GPU T4, tự động chuyển FP16 nếu detect T4

## 0. Tham số chính

In [ ]:
# Config chính
YAML_CONFIG = "train/Uni-MuMER-train.yaml"
MODEL_DIR = "Uni-MuMER-Qwen2.5-VL-3B"

# DagsHub MLflow Tracking
DAGSHUB_USERNAME = "NhatPot"  # Thay bằng username DagsHub của bạn
DAGSHUB_REPO = "test-unimer"  # Thay bằng tên repo DagsHub
MLFLOW_TRACKING_URI = f"https://dagshub.com/{DAGSHUB_USERNAME}/{DAGSHUB_REPO}.mlflow"

# Token: Khuyên dùng Kaggle Secrets (Add-ons -> Secrets -> DAGSHUB_TOKEN)
DAGSHUB_TOKEN = "YOUR_DAGSHUB_TOKEN"  # Hoặc lấy từ Kaggle Secrets

print(f"MLflow URI: {MLFLOW_TRACKING_URI}")

## 1. Cài Miniconda

In [ ]:
!rm -rf /kaggle/working/*
!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
!bash Miniconda3-latest-Linux-x86_64.sh -b -f -p /kaggle/working/miniconda
!rm Miniconda3-latest-Linux-x86_64.sh

import os
os.environ['PATH'] = "/kaggle/working/miniconda/bin:" + os.environ['PATH']

## 2. Tạo env Python 3.10

In [ ]:
!/kaggle/working/miniconda/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main -q
!/kaggle/working/miniconda/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r -q
!/kaggle/working/miniconda/bin/conda create -n unimumer python=3.10 -y -q

In [ ]:
# Check Python version
!source /kaggle/working/miniconda/bin/activate unimumer && python --version

## 3. Clone repo

In [ ]:
%cd /kaggle/working
!git clone https://github.com/NhatPot/test-unimer.git
%cd test-unimer
!pwd

## 4. Setup DagsHub credentials

In [ ]:
import os

# Lấy token từ Kaggle Secrets (ưu tiên) hoặc biến phía trên
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    token = secrets.get_secret("DAGSHUB_TOKEN")
    username = secrets.get_secret("DAGSHUB_USERNAME") if "DAGSHUB_USERNAME" in dir(secrets) else DAGSHUB_USERNAME
    repo = secrets.get_secret("DAGSHUB_REPO_NAME") if "DAGSHUB_REPO_NAME" in dir(secrets) else DAGSHUB_REPO
    print("✅ Using Kaggle Secrets")
except:
    token = DAGSHUB_TOKEN
    username = DAGSHUB_USERNAME
    repo = DAGSHUB_REPO
    print("⚠️  Using hardcoded token (add DAGSHUB_TOKEN to Kaggle Secrets)")

# Set env
os.environ["MLFLOW_TRACKING_URI"] = f"https://dagshub.com/{username}/{repo}.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"] = username
os.environ["MLFLOW_TRACKING_PASSWORD"] = token
os.environ["DAGSHUB_TOKEN"] = token
os.environ["DAGSHUB_USERNAME"] = username
os.environ["DAGSHUB_REPO_NAME"] = repo

print(f"MLflow: {os.environ['MLFLOW_TRACKING_URI']}")
print(f"User: {username}, Repo: {repo}")

## 5. Download base model

In [ ]:
!mkdir -p {MODEL_DIR}
!wget -q --show-progress -L -O {MODEL_DIR}/model-00001-of-00002.safetensors \
  "https://huggingface.co/phxember/Uni-MuMER-Qwen2.5-VL-3B/resolve/main/model-00001-of-00002.safetensors?download=true"
!wget -q --show-progress -L -O {MODEL_DIR}/model-00002-of-00002.safetensors \
  "https://huggingface.co/phxember/Uni-MuMER-Qwen2.5-VL-3B/resolve/main/model-00002-of-00002.safetensors?download=true"
!ls -lh {MODEL_DIR}/*.safetensors

## 6. Install dependencies

In [ ]:
ACTIVATE = "source /kaggle/working/miniconda/bin/activate unimumer"
!{ACTIVATE} && pip install -q -r requirements.txt

In [ ]:
%cd train/LLaMA-Factory
!{ACTIVATE} && pip install -q -e .
%cd ../..

## 7. Check GPU & auto-config

In [ ]:
import subprocess, yaml

# Check GPU (wrap in try-except for local testing)
is_t4 = False
try:
    !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], 
                              capture_output=True, text=True, check=False)
    if gpu_info.returncode == 0:
        gpu_name = gpu_info.stdout.strip()
        is_t4 = "T4" in gpu_name
        print(f"GPU: {gpu_name}")
except:
    print("⚠️  No GPU detected (OK for local testing)")

# Auto fix config cho T4
with open(YAML_CONFIG, "r") as f:
    config = yaml.safe_load(f)

if is_t4:
    config["bf16"] = False
    config["fp16"] = True
    print("⚠️  T4 detected → FP16 enabled")
else:
    print("ℹ️  Non-T4 or no GPU, keeping config as-is")

config["report_to"] = "none"  # Log thủ công sau

with open(YAML_CONFIG, "w") as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

print(f"✅ Config: bf16={config["bf16"]}, fp16={config["fp16"]}")

## 8. Init DagsHub

In [ ]:
# Import từ env unimumer (đã cài dagshub, mlflow)
!{ACTIVATE} && python -c "import dagshub; print('dagshub version:', dagshub.__version__)"

# Vì kernel hiện tại không có dagshub, ta sẽ gọi trong training script
# Cell này chỉ verify env đã cài đúng
print("✅ DagsHub will be initialized in training script")
print(f"   MLflow URI: {os.environ['MLFLOW_TRACKING_URI']}")

## 9. Training

In [ ]:
!{ACTIVATE} && MPLBACKEND=Agg llamafactory-cli train {YAML_CONFIG}

## 10. Upload to DagsHub

In [ ]:
from pathlib import Path
from datetime import datetime

# Load config
with open(YAML_CONFIG, 'r') as f:
    config = yaml.safe_load(f)

output_dir = config['output_dir']
checkpoints = sorted(Path(output_dir).glob("checkpoint-*"))
latest_checkpoint = str(checkpoints[-1]) if checkpoints else output_dir

# Generate run name
dataset = config.get('dataset', '')
if 'error' in str(dataset).lower():
    task = "EDL"
elif 'tree' in str(dataset).lower():
    task = "TreeCoT"
elif '_can' in str(dataset).lower():
    task = "SymbolCount"
else:
    task = "Train"

run_name = f"{task}_rank{config.get('lora_rank',64)}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

print(f"Output: {output_dir}")
print(f"Checkpoint: {latest_checkpoint}")
print(f"Run: {run_name}")
!du -sh {latest_checkpoint}

In [ ]:
# Upload tất cả lên DagsHub
!{ACTIVATE} && python scripts/post_training_logger.py \
  --yaml-config {YAML_CONFIG} \
  --output-dir {output_dir} \
  --checkpoint-dir {latest_checkpoint} \
  --run-name "{run_name}" \
  --dataset-info train/dataset_info.json

## 11. Verify & create backup

In [ ]:
# Check files
!echo "===== Important files ====="
!ls -lh {latest_checkpoint}/adapter_model.safetensors
!ls -lh {output_dir}/README_RUN.md
!ls -lh {output_dir}/ARTIFACT_MANIFEST.json

In [ ]:
# Create backup archive
archive = f"unimumer_{task}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.tar.gz"
!tar -czf /kaggle/working/{archive} -C {output_dir} .
!ls -lh /kaggle/working/{archive}

## 12. Summary

In [ ]:
print("="*60)
print("✅ TRAINING COMPLETE")
print("="*60)
print(f"\n📁 Local: {output_dir}")
print(f"📁 Checkpoint: {latest_checkpoint}")
print(f"📦 Archive: {archive}")
print(f"\n☁️  DagsHub: https://dagshub.com/{os.environ['DAGSHUB_USERNAME']}/{os.environ['DAGSHUB_REPO_NAME']}")
print(f"📊 Experiments: Click 'Experiments' tab → Find run '{run_name}'")
print(f"\n💾 Upload: Logs + Checkpoint (~643MB) uploaded to DagsHub")
print(f"📥 Download: Go to MLflow UI → Artifacts → Download")
print("="*60)